In [2]:
import os
import numpy as np
import tensorflow as tf
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Input, Dense, Conv2D, Flatten
from tensorflow.keras.layers import MaxPooling2D,  BatchNormalization, Activation
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()
import warnings
warnings.filterwarnings("ignore", category=Warning)


# Helper functions

In [3]:

def filter_label(df, y, label):
    # Splits the dataset into two parts based on the specified label.
    # Returns the subset of the dataframe and labels matching the label,
    # and the subset of the dataframe and labels not matching the label.
     
    output_df = df[np.where(y == label)[0]]
    
    res_df =  df[np.where(y != label)[0]]
    
    output_y = y[np.where(y == label)[0]]
    res_y = y[np.where(y != label)[0]]
    
    return  output_df, output_y, res_df, res_y

 


def obtain_labels(df, label_path):
    # Maps feature data from the dataframe to their corresponding labels from a CSV file.
    # Returns arrays of features and their corresponding labels.
    
    labels = pd.read_csv(label_path, header=0)
    
    y = []
    x = []
    
    for index, row in df.iterrows():
        #print(row["asm_id"].split(".")[0])
        hash_id = row["asm_id"].split(".")[0]
        if hash_id in labels['asm_id'].values: 
            row = row.drop("asm_id")
            x.append(row)
            y.append(labels[labels["asm_id"] == hash_id]["Class"])
            

    
    return np.array(x), np.array(y)


def change_attack_label(x):
    # Changes the attack label to 1.
    # Returns a list with a single element [1.].
    label = [1.]
    return label


# Load malware images
def load_image_malware(image_path, label_path):
    
    labels = pd.read_csv(label_path, header=0)
    
    x = []
    y = []
            
    
    
    for filename in os.listdir(image_path):
        if filename.endswith(".png"):
            hash_id = filename.split(".")[0]
            if hash_id in labels['asm_id'].values: 
                f = os.path.join(image_path, filename)
                image = Image.open(f).convert('RGB')
                image = image.resize((56, 56), Image.ANTIALIAS)
                image = np.array(image, dtype=int)
                x.append(image)
                y.append(labels[labels["asm_id"] == hash_id]["Class"])

         
            
    x = np.asarray(x)
    y = np.asarray(y)
                       
    x = x.astype('float32') / 255.
    
    
        
    return x, y 



# Load normal images

def load_image_normal(directory_path):
    image_list = []
    image_size_limit = 178956970  # Maximum allowed pixels per image

    for filename in os.listdir(directory_path):
        if filename.endswith(".jpg") or filename.endswith(".png") or filename.endswith(".jpeg"):
            file_path = os.path.join(directory_path, filename)
            try:
                Image.MAX_IMAGE_PIXELS = None  # Disable DecompressionBombError check for large images
                with Image.open(file_path) as img:
                    Image.MAX_IMAGE_PIXELS = None  # Disable DecompressionBombError check for large images
                    # Check if the image size is within the allowed limit
                    if img.width * img.height <= image_size_limit:
                        img = img.convert('RGB')
                        img = img.resize((56, 56), Image.ANTIALIAS)
                        img_array = np.array(img, dtype=int)
                        image_list.append(img_array)
                    else:
                        print(f"Image {filename} exceeds the size limit of {image_size_limit} pixels and will be skipped.")
            except (Image.DecompressionBombError, OSError) as e:
                print(f"Error loading image {filename}: {e}")

    image_list = np.asarray(image_list)
    image_list = image_list.astype('float32') / 255.

    return image_list



# Load malware data

In [4]:
#It will take some time to load the malware images

label_path = "data/labels/new_gmm_labels_id.csv"
img_path ="data/image_features/Big15_2"
malware_x, malware_y =load_image_malware(img_path, label_path)
print(malware_x.shape)
print(malware_y.shape)

(10711, 56, 56, 3)
(10711, 1)


# Load benign data source

In [7]:
img_path ="data/image_features/benign_source/dataset1"
source_normal_x_1 =load_image_normal(img_path)
source_normal_y_1 = np.zeros((source_normal_x_1.shape[0],1))

img_path ="data/image_features/benign_source/dataset2"
source_normal_x_2 =load_image_normal(img_path)
source_normal_y_2 = np.zeros((source_normal_x_2.shape[0],1))

img_path ="data/image_features/benign_source/dataset3"
source_normal_x_3=load_image_normal(img_path)
source_normal_y_3 = np.zeros((source_normal_x_3.shape[0],1))

img_path ="data/image_features/benign_source/dataset4"
source_normal_x_4 =load_image_normal(img_path)
source_normal_y_4 = np.zeros((source_normal_x_4.shape[0],1))

#merge
source_normal_x = np.concatenate((source_normal_x_1, source_normal_x_2, source_normal_x_3, source_normal_x_4), axis = 0)
source_normal_y = np.concatenate((source_normal_y_1, source_normal_y_2, source_normal_y_3, source_normal_y_4), axis = 0)
print(source_normal_x.shape)
print(source_normal_y.shape)


Image win11_DropboxClientInstaller_exe.png exceeds the size limit of 178956970 pixels and will be skipped.
Image win11_mbuns_exe.png exceeds the size limit of 178956970 pixels and will be skipped.
(8208, 56, 56, 3)
(8208, 1)


# Load benign data target

In [6]:
img_path ="data/image_features/benign_target/dataset1"
target_normal_x_1 =load_image_normal(img_path)
target_normal_y_1 = np.zeros((target_normal_x_1.shape[0],1))

img_path ="data/image_features/benign_target/dataset2"
target_normal_x_2 =load_image_normal(img_path)
target_normal_y_2 = np.zeros((target_normal_x_2.shape[0],1))

img_path ="data/image_features/benign_target/dataset3"
target_normal_x_3=load_image_normal(img_path)
target_normal_y_3 = np.zeros((target_normal_x_3.shape[0],1))

img_path ="data/image_features/benign_target/dataset4"
target_normal_x_4 =load_image_normal(img_path)
target_normal_y_4 = np.zeros((target_normal_x_4.shape[0],1))

#merge
target_normal_x = np.concatenate((target_normal_x_1, target_normal_x_2, target_normal_x_3, target_normal_x_4), axis = 0)
target_normal_y = np.concatenate((target_normal_y_1, target_normal_y_2, target_normal_y_3, target_normal_y_4), axis = 0)
print(target_normal_x.shape)
print(target_normal_y.shape)

Image gimp-2_10_18-setup-1_exe.png exceeds the size limit of 178956970 pixels and will be skipped.
Image digiKam-7_0_0-Win32_exe.png exceeds the size limit of 178956970 pixels and will be skipped.
Image iTunesSetup_exe.png exceeds the size limit of 178956970 pixels and will be skipped.
(9150, 56, 56, 3)
(9150, 1)


# Models

In [8]:
class DANN_CNN(object):
    def __init__(self, x_source_train, y_source_train, 
                 x_target_train, y_target_train, 
                 x_source_test, y_source_test, 
                 x_target_test, y_target_test, 
                 epochs=90):

        #source train and test dataset
        self.x_source_train = x_source_train
        self.y_source_train = y_source_train
        
        self.x_source_test = x_source_test
        self.y_source_test = y_source_test
        
        # Target train and test dataset
        
        self.x_target_train = x_target_train
        self.y_target_train = y_target_train

        self.x_target_test = x_target_test
        self.y_target_test = y_target_test


        self.n_classes = y_source_train.shape[1]
        
         
        # Use the source dataset shape for the generator input and outputs.
        self.input_shape = x_source_train.shape[1:]
        self.output_shape = y_source_train.shape[1:]
        

        
        
       
        self.epochs = epochs 


        self.generator = Sequential([
            Input(shape=(56,56,3)),
            Conv2D(32, kernel_size=(3, 3), activation="relu"),
            MaxPooling2D(pool_size=(2, 2)),
            Conv2D(64, kernel_size=(3, 3), activation="relu"),
            MaxPooling2D(pool_size=(2, 2)),
            Flatten(),
            #Dropout(0.5)
            #Dense(num_classes, activation="softmax"),
        ])
    

        self.classifier = Sequential([
            #Conv2D(filters=2048, kernel_size=(1,1)),
            #GlobalAveragePooling2D(),
            Dense(256, activation = "relu"),
            Dense(2, activation = "softmax")
        ])
        
        self.discriminator = Sequential([
           # Conv2D(filters=2048, kernel_size=(1,1)),
            #GlobalAveragePooling2D(),
            Dense(1024),
            BatchNormalization(),
            Activation('relu'),
            Dense(1024),
            BatchNormalization(),
            Activation('relu'), 
            Dense(2, activation='softmax')
        ])
        
        
        
    

      
    
        
      
        self.loss = tf.keras.losses.CategoricalCrossentropy()

       

    
        
        self.lr = 0.001 
        self.momentum = 0.9
        self.alpha = 0.0002


        self.task_optimizer= Adam(0.001)
        self.gen_optimizer = Adam(0.001)
        self.disc_optimizer = Adam(0.001)
       
        
        
        self.train_task_loss = tf.keras.metrics.Mean()
        self.train_target_task_loss = tf.keras.metrics.Mean()
        self.train_disc_loss = tf.keras.metrics.Mean()
        self.train_gen_loss = tf.keras.metrics.Mean()
        

        self.train_task_accuracy = tf.keras.metrics.CategoricalAccuracy()
        self.train_target_task_accuracy = tf.keras.metrics.CategoricalAccuracy()
        self.train_domain_accuracy = tf.keras.metrics.CategoricalAccuracy()


        self.test_task_loss = tf.keras.metrics.Mean()
        self.test_disc_loss = tf.keras.metrics.Mean()
        self.test_gen_loss = tf.keras.metrics.Mean()
        self.test_target_task_loss = tf.keras.metrics.Mean()
    

        self.test_task_accuracy = tf.keras.metrics.CategoricalAccuracy()
        self.test_target_task_accuracy = tf.keras.metrics.CategoricalAccuracy()
        

        
        self.batch_size = 32



    def train_batch(self, x_source_train, y_source_train, x_target_train, y_target_train, epoch):
        
        source = np.tile([1,0], (x_source_train.shape[0], 1))
        target = np.tile([0,1], (x_target_train.shape[0], 1))
        domain_labels = np.concatenate([source, target], axis = 0)
        class_labels = np.concatenate([y_source_train, y_target_train], axis = 0)
        


        x_both = tf.concat([x_source_train, x_target_train], axis = 0)
        
        
        with tf.GradientTape() as disc_tape:
            
            #Forward pa
            y_domain_pred = self.discriminator(self.generator(x_both, training=True), training =True)
            disc_loss = self.loss(domain_labels, y_domain_pred)  
          
        
        # Compute gradients   
        disc_grad = disc_tape.gradient(disc_loss, self.discriminator.trainable_variables) 
        
        # Update weights 
        self.disc_optimizer.apply_gradients(zip(disc_grad, self.discriminator.trainable_variables))
    
        self.train_disc_loss(disc_loss)
       

        with tf.GradientTape() as task_tape, tf.GradientTape() as gen_tape:
            
            #Forward pass
            y_class_pred = self.classifier(self.generator(x_source_train, training=True), training =True)
            y_class_pred_target =  self.classifier(self.generator(x_target_train, training=True), training =True)
            y_domain_pred = self.discriminator(self.generator(x_both, training=True), training =True)
        
            task_loss_source = self.loss(y_source_train, y_class_pred) 
            task_loss_target = self.loss(y_target_train, y_class_pred_target)
            
            task_loss = task_loss_source + task_loss_target
            disc_loss = self.loss(domain_labels, y_domain_pred)  
            
        
            gen_loss = task_loss - disc_loss * 0.1
           
            
            
        
        
         # Compute gradients   
        task_grad = task_tape.gradient(task_loss, self.classifier.trainable_variables)
        gen_grad = gen_tape.gradient(gen_loss, self.generator.trainable_variables)
        #disc_grad = disc_tape.gradient(disc_loss, self.discriminator.trainable_variables) 
        
        
        # Update weights 
        self.task_optimizer.apply_gradients(zip(task_grad, self.classifier.trainable_variables))
        self.gen_optimizer.apply_gradients(zip(gen_grad, self.generator.trainable_variables)) 
        #self.disc_optimizer.apply_gradients(zip(disc_grad, self.discriminator.trainable_variables))
        
        self.train_task_loss(task_loss_source)
        self.train_task_accuracy(y_source_train, y_class_pred)
        
        self.train_target_task_loss(task_loss_target)
        self.train_target_task_accuracy(y_target_train, y_class_pred_target)
        
        self.train_gen_loss(gen_loss)
       

        return
    
    def test_batch(self, x_source_test, y_source_test, x_target_test, y_target_test):
        
 
        
        
        with tf.GradientTape() as tape:
            
            DIrep_source = self.generator(x_source_test, training=False)
            DIrep_target = self.generator(x_target_test, training=False)
            
            y_class_pred = self.classifier(DIrep_source, training=False)
            y_target_class_pred = self.classifier(DIrep_target, training=False)
            
    
            task_loss = self.loss(y_source_test, y_class_pred)
            target_task_loss = self.loss(y_target_test, y_target_class_pred)
           
            

        self.test_task_loss(task_loss)
        self.test_task_accuracy(y_source_test, y_class_pred)
       


        self.test_target_task_loss(target_task_loss)
        self.test_target_task_accuracy(y_target_test, y_target_class_pred)

        return
    
    def test(self, x_source_test, y_source_test, x_target_test, y_target_test):
        
        with tf.GradientTape() as tape:
            y_class_pred = self.predict_label(x_source_test, training=False)
            y_target_class_pred = self.predict_label(x_target_test, training=False)
            
    
            task_loss = self.loss(y_source_test, y_class_pred)
            target_task_loss = self.loss(y_target_test, y_target_class_pred)
           
            

        self.test_task_loss(task_loss)
        self.test_task_accuracy(y_source_test, y_class_pred)
        
        
   

        self.test_target_task_loss(target_task_loss)
        self.test_target_task_accuracy(y_target_test, y_target_class_pred)

        return
    
    def log(self):
        
        
        log_format = 'c_loss source train: {:.4f}, acc source train : {:.2f}\n'+ \
            'c_loss target train: {:.4f}, acc target train : {:.2f}\n'+ \
            'D_loss train: {:.4f}\n'+ \
            'C_loss test source: {:.4f}, Acc test source: {:.2f}\n'+ \
            'C_loss test target: {:.4f}, Acc test target: {:.2f}\n'

        message = log_format.format(
                 self.train_task_loss.result(),
                 self.train_task_accuracy.result()*100,
                 self.train_target_task_loss.result(),
                 self.train_target_task_accuracy.result()*100,
                 self.train_disc_loss.result(),
                 self.test_task_loss.result(),
                 self.test_task_accuracy.result()*100,
                 self.test_target_task_loss.result(),
                 self.test_target_task_accuracy.result()*100)
        

        self.reset_metrics('train')
        self.reset_metrics('test')


        return message 
    
    def reset_metrics(self, target):

        if target == 'train':
            self.train_task_loss.reset_states()
            self.train_task_accuracy.reset_states()
            self.train_target_task_loss.reset_states()
            self.train_target_task_accuracy.reset_states()
            self.train_disc_loss.reset_states()
           
        
        if target == 'test':
            self.test_task_loss.reset_states()
            self.test_task_accuracy.reset_states()
            self.test_target_task_loss.reset_states()
            self.test_target_task_accuracy.reset_states()



        return
    
    
    def train(self):
        
        source_train_dataset = tf.data.Dataset.from_tensor_slices((self.x_source_train, self.y_source_train)).shuffle(len(self.y_source_train)).batch(self.batch_size)
        target_train_dataset = tf.data.Dataset.from_tensor_slices((self.x_target_train, self.y_target_train)).shuffle(len(self.y_target_train)).batch(self.batch_size)
        
        source_test_dataset = tf.data.Dataset.from_tensor_slices((self.x_source_test, self.y_source_test)).batch(self.batch_size)
        target_test_dataset = tf.data.Dataset.from_tensor_slices((self.x_target_test, self.y_target_test)).batch(self.batch_size)
        

        
        
        for epoch in range(self.epochs):
            
            batches = 0 
            
            for (source_images, source_labels), (target_images, target_labels) in zip(source_train_dataset, target_train_dataset):
                self.train_batch(source_images, source_labels, target_images, target_labels, epoch)
            


                

            for (test_images, test_labels), (target_test_images, target_test_labels) in zip(source_test_dataset, target_test_dataset):
                self.test_batch(test_images, test_labels, target_test_images, target_test_labels)
            
  
            print('Epoch: {}'.format(epoch + 1))
            print(self.log())
            
        return self.generator, self.classifier

    
    
    
    
    

# Training 

## Set Cluster 0 as the target domain

In [9]:

target_malware_x, target_malware_y, source_malware_x, source_malware_y =  filter_label(malware_x, malware_y, [0])
print("Malware data ...")
print("Target: {}".format(target_malware_x.shape))
print("Target; {}".format(target_malware_y.shape))
print("Source {}".format(source_malware_x.shape))
print("Source {}".format(source_malware_y.shape))

print("Normal data ...")
print("Target: {}".format(target_normal_x.shape))
print("Target: {}".format(target_normal_y.shape))
print("Source {}".format(source_normal_x.shape))
print("Source {}".format(source_normal_y.shape))

#concatenate source and target
source_malware_y = np.apply_along_axis(change_attack_label, 1, source_malware_y)
target_malware_y = np.apply_along_axis(change_attack_label, 1, target_malware_y)


source_x = np.concatenate((source_malware_x, source_normal_x), axis = 0)
source_y = np.concatenate((source_malware_y, source_normal_y), axis = 0)


target_x = np.concatenate((target_malware_x, target_normal_x), axis = 0)
target_y = np.concatenate((target_malware_y, target_normal_y), axis = 0)


#one-hot encode labels

source_y = tf.keras.utils.to_categorical(source_y, num_classes = 2)
target_y = tf.keras.utils.to_categorical(target_y, num_classes = 2)




print("Combined data ...")
print("Target: {}".format(target_x.shape))
print("Target: {}".format(target_y.shape))
print("Source {}".format(source_x.shape))
print("Source {}".format(source_y.shape))



#split data into train and test
source_x_train, source_x_test, source_y_train, source_y_test = train_test_split(source_x, source_y, test_size=0.25, random_state=42)
target_x_train, target_x_test, target_y_train, target_y_test = train_test_split(target_x, target_y, test_size=0.5, random_state=42)
        
    

print("train test data ...")
print("Target train: {}".format(target_x_train.shape))
print("Target train: {}".format(target_y_train.shape))
print("Target test: {}".format(target_x_test.shape))
print("Target test: {}".format(target_y_test.shape))
print("Source train: {}".format(source_x_train.shape))
print("Source train: {}".format(source_y_train.shape))
print("Source test: {}".format(source_x_test.shape))
print("Source test: {}".format(source_y_test.shape))



           

Malware data ...
Target: (6023, 56, 56, 3)
Target; (6023, 1)
Source (4688, 56, 56, 3)
Source (4688, 1)
Normal data ...
Target: (9150, 56, 56, 3)
Target: (9150, 1)
Source (8208, 56, 56, 3)
Source (8208, 1)
Combined data ...
Target: (15173, 56, 56, 3)
Target: (15173, 2)
Source (12896, 56, 56, 3)
Source (12896, 2)
train test data ...
Target train: (7586, 56, 56, 3)
Target train: (7586, 2)
Target test: (7587, 56, 56, 3)
Target test: (7587, 2)
Source train: (9672, 56, 56, 3)
Source train: (9672, 2)
Source test: (3224, 56, 56, 3)
Source test: (3224, 2)


In [11]:
samples = [20, 50, 100, 200, 300, 500]
################################################################################
# Config
################################################################################
#learning_rate_1 = 1e-3  # Learning rate
learning_rate_2 = 0.0001

# We have limited the number of training epochs to 20 to minimize training time.
# You can change the epochs to a lower number (lowest 1) just to test if the code is functional.
# However, for better model performance, consider increasing the number of epochs to those specified in the referenced paper.
epochs = 20  
n_class = 2 
input_shape  = source_x.shape[1]




for size in samples: 
    print("--------------------------Sample size {}-------------------------".format(size))
    
    # we randomly select 3000 data from the source training set to reduce the amount of time + GPU memory required for training
    idxs = np.random.permutation(source_x_train.shape[0]) 
    split = int(3000)
    idx_sample, _= np.split(idxs, [split])
    source_x_train_select =  source_x_train[idx_sample]
    source_y_train_select =  source_y_train[idx_sample]
    print("source_x_train_select: {}".format(source_x_train_select.shape))
    print("source_y_train_select: {}".format(source_y_train_select.shape))

    # we randomly select different number of data from the target training set
    idxs = np.random.permutation(target_x_train.shape[0]) 
    split = int(size)
    idx_sample, _= np.split(idxs, [split])
    target_x_train_select =  target_x_train[idx_sample]
    target_y_train_select =  target_y_train[idx_sample]
    print("target_x_train_select: {}".format(target_x_train_select.shape))
    print("target_y_train_select: {}".format(target_y_train_select.shape))



    model =DANN_CNN(source_x_train_select, source_y_train_select, target_x_train_select, target_y_train_select,
    source_x_test, source_y_test, target_x_test, target_y_test, epochs=20) # change the epochs here as well

    generator, classifier = model.train()

    y_target_class_pred = classifier.predict(generator(target_x_test)).argmax(1)

    result = accuracy_score(target_y_test.argmax(1), y_target_class_pred)

    print("The test acc is {}".format(result))



    
    

--------------------------Sample size 20-------------------------
source_x_train_select: (3000, 56, 56, 3)
source_y_train_select: (3000, 2)
target_x_train_select: (20, 56, 56, 3)
target_y_train_select: (20, 2)
Epoch: 1
c_loss source train: 0.7436, acc source train : 31.25
c_loss target train: 0.7041, acc target train : 50.00
D_loss train: 1.2689
C_loss test source: 1.2171, Acc test source: 62.87
C_loss test target: 1.2899, Acc test target: 59.59

Epoch: 2
c_loss source train: 1.1152, acc source train : 65.62
c_loss target train: 1.9973, acc target train : 45.00
D_loss train: 1.2493
C_loss test source: 0.6221, Acc test source: 62.87
C_loss test target: 0.6449, Acc test target: 59.59

Epoch: 3
c_loss source train: 0.6586, acc source train : 56.25
c_loss target train: 0.8397, acc target train : 45.00
D_loss train: 0.6991
C_loss test source: 0.7158, Acc test source: 39.42
C_loss test target: 0.7270, Acc test target: 38.83

Epoch: 4
c_loss source train: 0.7167, acc source train : 37.50
c_lo

## Set Cluster 1 as the target domain

In [12]:

target_malware_x, target_malware_y, source_malware_x, source_malware_y =  filter_label(malware_x, malware_y, [1])
print("Malware data ...")
print("Target: {}".format(target_malware_x.shape))
print("Target; {}".format(target_malware_y.shape))
print("Source {}".format(source_malware_x.shape))
print("Source {}".format(source_malware_y.shape))

print("Normal data ...")
print("Target: {}".format(target_normal_x.shape))
print("Target: {}".format(target_normal_y.shape))
print("Source {}".format(source_normal_x.shape))
print("Source {}".format(source_normal_y.shape))

#concatenate source and target
source_malware_y = np.apply_along_axis(change_attack_label, 1, source_malware_y)
target_malware_y = np.apply_along_axis(change_attack_label, 1, target_malware_y)



source_x = np.concatenate((source_malware_x, source_normal_x), axis = 0)
source_y = np.concatenate((source_malware_y, source_normal_y), axis = 0)


target_x = np.concatenate((target_malware_x, target_normal_x), axis = 0)
target_y = np.concatenate((target_malware_y, target_normal_y), axis = 0)


#one-hot encode labels

source_y = tf.keras.utils.to_categorical(source_y, num_classes = 2)
target_y = tf.keras.utils.to_categorical(target_y, num_classes = 2)




print("Combined data ...")
print("Target: {}".format(target_x.shape))
print("Target: {}".format(target_y.shape))
print("Source {}".format(source_x.shape))
print("Source {}".format(source_y.shape))



#split data into train and test
source_x_train, source_x_test, source_y_train, source_y_test = train_test_split(source_x, source_y, test_size=0.25, random_state=42)
target_x_train, target_x_test, target_y_train, target_y_test = train_test_split(target_x, target_y, test_size=0.5, random_state=42)
        
    

print("train test data ...")
print("Target train: {}".format(target_x_train.shape))
print("Target train: {}".format(target_y_train.shape))
print("Target test: {}".format(target_x_test.shape))
print("Target test: {}".format(target_y_test.shape))
print("Source train: {}".format(source_x_train.shape))
print("Source train: {}".format(source_y_train.shape))
print("Source test: {}".format(source_x_test.shape))
print("Source test: {}".format(source_y_test.shape))



           

Malware data ...
Target: (3206, 56, 56, 3)
Target; (3206, 1)
Source (7505, 56, 56, 3)
Source (7505, 1)
Normal data ...
Target: (9150, 56, 56, 3)
Target: (9150, 1)
Source (8208, 56, 56, 3)
Source (8208, 1)
Combined data ...
Target: (12356, 56, 56, 3)
Target: (12356, 2)
Source (15713, 56, 56, 3)
Source (15713, 2)
train test data ...
Target train: (6178, 56, 56, 3)
Target train: (6178, 2)
Target test: (6178, 56, 56, 3)
Target test: (6178, 2)
Source train: (11784, 56, 56, 3)
Source train: (11784, 2)
Source test: (3929, 56, 56, 3)
Source test: (3929, 2)


In [13]:
samples = [20, 50, 100, 200, 300, 500]
################################################################################
# Config
################################################################################
#learning_rate_1 = 1e-3  # Learning rate
learning_rate_2 = 0.0001
# We have limited the number of training epochs to 20 to minimize training time.
# You can change the epochs to a lower number (lowest 1) just to test if the code is functional.
# However, for better model performance, consider increasing the number of epochs to those specified in the referenced paper. 
epochs = 20  # Number of training epochs
n_class = 2 
input_shape  = source_x.shape[1]




for size in samples: 
    print("--------------------------Sample size {}-------------------------".format(size))
    
     # we randonly select 3000 data from the source training set to reduce the amount of time + GPU memory required for training
    idxs = np.random.permutation(source_x_train.shape[0]) 
    split = int(3000)
    idx_sample, _= np.split(idxs, [split])
    source_x_train_select =  source_x_train[idx_sample]
    source_y_train_select =  source_y_train[idx_sample]
    print("source_x_train_select: {}".format(source_x_train_select.shape))
    print("source_y_train_select: {}".format(source_y_train_select.shape))

    # we randomly select different number of data from the target training set
    idxs = np.random.permutation(target_x_train.shape[0]) 
    split = int(size)
    idx_sample, _= np.split(idxs, [split])
    target_x_train_select =  target_x_train[idx_sample]
    target_y_train_select =  target_y_train[idx_sample]
    print("target_x_train_select: {}".format(target_x_train_select.shape))
    print("target_y_train_select: {}".format(target_y_train_select.shape))




    model =DANN_CNN(source_x_train_select, source_y_train_select, target_x_train_select, target_y_train_select,
    source_x_test, source_y_test, target_x_test, target_y_test, epochs=20) # change the epochs here as well


    generator, classifier = model.train()

    y_target_class_pred = classifier.predict(generator(target_x_test)).argmax(1)

    result = accuracy_score(target_y_test.argmax(1), y_target_class_pred)

    print("The test acc is {}".format(result))

        
    
    
    

--------------------------Sample size 20-------------------------
source_x_train_select: (3000, 56, 56, 3)
source_y_train_select: (3000, 2)
target_x_train_select: (20, 56, 56, 3)
target_y_train_select: (20, 2)
Epoch: 1
c_loss source train: 0.6874, acc source train : 46.88
c_loss target train: 0.6901, acc target train : 55.00
D_loss train: 1.0375
C_loss test source: 1.2833, Acc test source: 52.13
C_loss test target: 0.9641, Acc test target: 73.91

Epoch: 2
c_loss source train: 1.4339, acc source train : 46.88
c_loss target train: 1.4661, acc target train : 65.00
D_loss train: 2.0478
C_loss test source: 0.6247, Acc test source: 74.14
C_loss test target: 0.6129, Acc test target: 75.53

Epoch: 3
c_loss source train: 0.6420, acc source train : 68.75
c_loss target train: 0.6300, acc target train : 65.00
D_loss train: 0.9313
C_loss test source: 0.9319, Acc test source: 47.87
C_loss test target: 1.2581, Acc test target: 26.09

Epoch: 4
c_loss source train: 1.0264, acc source train : 40.62
c_lo

## Set Cluster 2 as the target domain

In [14]:

target_malware_x, target_malware_y, source_malware_x, source_malware_y =  filter_label(malware_x, malware_y, [2])
print("Malware data ...")
print("Target: {}".format(target_malware_x.shape))
print("Target; {}".format(target_malware_y.shape))
print("Source {}".format(source_malware_x.shape))
print("Source {}".format(source_malware_y.shape))

print("Normal data ...")
print("Target: {}".format(target_normal_x.shape))
print("Target: {}".format(target_normal_y.shape))
print("Source {}".format(source_normal_x.shape))
print("Source {}".format(source_normal_y.shape))

#concatenate source and target
source_malware_y = np.apply_along_axis(change_attack_label, 1, source_malware_y)
target_malware_y = np.apply_along_axis(change_attack_label, 1, target_malware_y)



source_x = np.concatenate((source_malware_x, source_normal_x), axis = 0)
source_y = np.concatenate((source_malware_y, source_normal_y), axis = 0)



target_x = np.concatenate((target_malware_x, target_normal_x), axis = 0)
target_y = np.concatenate((target_malware_y, target_normal_y), axis = 0)


#one-hot encode labels

source_y = tf.keras.utils.to_categorical(source_y, num_classes = 2)
target_y = tf.keras.utils.to_categorical(target_y, num_classes = 2)




print("Combined data ...")
print("Target: {}".format(target_x.shape))
print("Target: {}".format(target_y.shape))
print("Source {}".format(source_x.shape))
print("Source {}".format(source_y.shape))




#split data into train and test
source_x_train, source_x_test, source_y_train, source_y_test = train_test_split(source_x, source_y, test_size=0.25, random_state=42)
target_x_train, target_x_test, target_y_train, target_y_test = train_test_split(target_x, target_y, test_size=0.5, random_state=42)
        
    

print("train test data ...")
print("Target train: {}".format(target_x_train.shape))
print("Target train: {}".format(target_y_train.shape))
print("Target test: {}".format(target_x_test.shape))
print("Target test: {}".format(target_y_test.shape))
print("Source train: {}".format(source_x_train.shape))
print("Source train: {}".format(source_y_train.shape))
print("Source test: {}".format(source_x_test.shape))
print("Source test: {}".format(source_y_test.shape))



           

Malware data ...
Target: (1438, 56, 56, 3)
Target; (1438, 1)
Source (9273, 56, 56, 3)
Source (9273, 1)
Normal data ...
Target: (9150, 56, 56, 3)
Target: (9150, 1)
Source (8208, 56, 56, 3)
Source (8208, 1)
Combined data ...
Target: (10588, 56, 56, 3)
Target: (10588, 2)
Source (17481, 56, 56, 3)
Source (17481, 2)
train test data ...
Target train: (5294, 56, 56, 3)
Target train: (5294, 2)
Target test: (5294, 56, 56, 3)
Target test: (5294, 2)
Source train: (13110, 56, 56, 3)
Source train: (13110, 2)
Source test: (4371, 56, 56, 3)
Source test: (4371, 2)


In [15]:
samples = [20, 50, 100, 200, 300, 500]
################################################################################
# Config
################################################################################
#learning_rate_1 = 1e-3  # Learning rate
learning_rate_2 = 0.0001
# We have limited the number of training epochs to 20 to minimize training time.
# You can change the epochs to a lower number (lowest 1) just to test if the code is functional.
# However, for better model performance, consider increasing the number of epochs to those specified in the referenced paper.
epochs = 20  # Number of training epochs
n_class = 2 
input_shape  = source_x.shape[1]




for size in samples: 
    print("--------------------------Sample size {}-------------------------".format(size))
        
    # we randonly select 3000 data from the source training set to reduce the amount of time + GPU memory required for training
    idxs = np.random.permutation(source_x_train.shape[0]) 
    split = int(3000)
    idx_sample, _= np.split(idxs, [split])
    source_x_train_select =  source_x_train[idx_sample]
    source_y_train_select =  source_y_train[idx_sample]
    print("source_x_train_select: {}".format(source_x_train_select.shape))
    print("source_y_train_select: {}".format(source_y_train_select.shape))

    # we randomly select different number of data from the target training set
    idxs = np.random.permutation(target_x_train.shape[0]) 
    split = int(size)
    idx_sample, _= np.split(idxs, [split])
    target_x_train_select =  target_x_train[idx_sample]
    target_y_train_select =  target_y_train[idx_sample]
    print("target_x_train_select: {}".format(target_x_train_select.shape))
    print("target_y_train_select: {}".format(target_y_train_select.shape))


    model =DANN_CNN(source_x_train_select, source_y_train_select, target_x_train_select, target_y_train_select,
    source_x_test, source_y_test, target_x_test, target_y_test, epochs=20) # change the epochs here as well


    generator, classifier = model.train()

    y_target_class_pred = classifier.predict(generator(target_x_test)).argmax(1)

    result = accuracy_score(target_y_test.argmax(1), y_target_class_pred)

    print("The test acc is {}".format(result))

        
    
    
    

--------------------------Sample size 20-------------------------
source_x_train_select: (3000, 56, 56, 3)
source_y_train_select: (3000, 2)
target_x_train_select: (20, 56, 56, 3)
target_y_train_select: (20, 2)
Epoch: 1
c_loss source train: 0.6835, acc source train : 59.38
c_loss target train: 0.7367, acc target train : 15.00
D_loss train: 0.9602
C_loss test source: 1.4131, Acc test source: 46.74
C_loss test target: 0.2285, Acc test target: 85.74

Epoch: 2
c_loss source train: 1.0135, acc source train : 68.75
c_loss target train: 0.3003, acc target train : 85.00
D_loss train: 1.6808
C_loss test source: 0.8607, Acc test source: 46.74
C_loss test target: 0.2792, Acc test target: 85.74

Epoch: 3
c_loss source train: 0.8914, acc source train : 37.50
c_loss target train: 0.2955, acc target train : 85.00
D_loss train: 2.0050
C_loss test source: 0.6552, Acc test source: 57.52
C_loss test target: 0.5100, Acc test target: 96.65

Epoch: 4
c_loss source train: 0.6565, acc source train : 56.25
c_lo